# HeatUp 01 - prepare

Puts every structure on disk and writes down **what is to be computed** for it. No calculator, no GPU.

| step | output (per `<material>/<label>/`) | used by | what it does |
|---|---|---|---|
| `relax` | `relaxation/CONTCAR`, `energy.json` | everything | BFGS (+ FrechetCellFilter) geometry optimisation |
| `elastic` | `elastic/elastic.json` (relaxed-ion) | Gate 1 | 6x6 Voigt stiffness tensor from strained cells |
| `phonons` | `phonons/ha.json` | Gates 2, 3 | harmonic modes (phonopy finite displacements); also gives the competing-phase free energies |
| `qha` (optional) | `phonons/qha.json` | Gate 3 with `hull.phonon_method="QHA"` | phonons at several volumes, F(V,T) fit |
| `fc3` (optional) | `phonons/fc3.json` | diagnostic | third-order force constants (phono3py): kappa_RTA(T), linewidths |
| `md` | `md/<T>K/output.traj` (positions, momenta, energy, stress) | Gates 0, 2; Gate 3 with `hull.phonon_method="VDOS"` (then also for the competing phases) | NPT or NVT trajectory in a supercell |

Everything below is a *calculation* setting. Together with the gates you ask for (`cfg.plan.gates`) and the steps they
need, it is written into a `plan.json` next to every POSCAR, so the folders describe themselves:

* the **folder of every planned calculation** is created right here (`relaxation/`, `elastic/`, `phonons/`, `md/<T>K/`), each with a
  `task_<step>.json` (inputs, files to write, options) - so a step can just as well be computed with **another code** (DFT, ...);
* **notebook 02** (optional) loads the interatomic potential and runs everything the plans ask for;
* **notebook 03** analyses whatever is on disk - also results you computed yourself (DFT, another potential).

The calculator, the gate thresholds and the analysis options are **not** set here.

## Where the data lives
```
<candidates_root>/<formula>/<label>/POSCAR
<database_root>/<material>/<label>/
    POSCAR
    plan.json
    metadata.json
    relaxation/
    elastic/
    phonons/
    md/<T>K/
    stability/
    <step dir>/manifest_<step>.json
    status_<step>.json
```
A structure is **copied** from candidates to the database the moment it is needed (as a target, or as a competing phase of a target).
Competing phases already in the database are reused - but only if they were calculated with the settings of this project;
otherwise they are left untouched, a warning is printed and they are not used.
`print_layout(root)` (end of this notebook) shows what exists. The full description is the docstring of
`heatup.core.layout`.

In [ ]:
import json
import os
import warnings
import pandas as pd
from IPython.display import display

import heatup
from heatup import cfg, prepare_project, preview_plan, save_project
from heatup.analyze.report import print_layout, show_status

warnings.filterwarnings("ignore")
heatup.verbose()
print("HeatUp", heatup.__version__)

HeatUp 0.4.0


## Configuration - the only cell you normally edit
Sections of `cfg` used here: `phases`, `plan`, `relax`, `elastic`, `phonon`, `fc3`, `qha`, `md` (and `generation`, `phases.mp_*` for competing phases).

In [ ]:
# -- The two folders ------------------------------------------------------------------------------------
cfg.phases.database_root = "database"  # everything computed: targets and competing phases
cfg.phases.candidates_root = "candidates"  # structures only (yours; MP downloads and generated ones are added here)
# Competing phases are searched in both folders; "mp-api" (needs cfg.phases.mp_api_key or $MP_API_KEY) and "generate" add structures
cfg.phases.sources = ["database", "candidates"]  # ["database", "candidates", "mp-api"]

# -- WHAT to compute: ask for what you want, everything it needs is planned (and notebook 02 runs all of it) ----
#   "relax"    -> optimise the structures. LEAVE IT OUT if they are already relaxed (other code / model / DFT): they are
#                 then used as given and only energy, forces and stress are evaluated (needed for the hull)
#   mechanical -> elastic | structural_transition -> md | vibrational -> phonons + md
#   thermodynamic -> phonons of the target AND of its competing phases
cfg.plan.gates = [
    "relax",
    "mechanical",
    "structural_transition",
    "vibrational",
    "thermodynamic"
    ]
# Free energies of the hull - the SAME method for every phase - decide what the competing phases need:
#   "HA"   harmonic phonons of every phase (default)                       -> competitors: relax + phonons
#   "QHA"  quasi-harmonic                                                  -> competitors: relax + qha
#   "VDOS" anharmonic: effective-harmonic F_vib from each phase's MD       -> competitors: relax + md (at the targets' temperatures)
# (the target's own MD, for Gates 0 and 2, is planned by the gates; notebook 03 follows this choice unless it is overridden there)
cfg.hull.phonon_method = "VDOS"          # None = "HA"
# Optional: list the steps yourself instead (relax | elastic | phonons | qha | fc3 | md; relax is always added),
# e.g. to add "qha" / "fc3", or "md" for competitors (VDOS free energies). None = derived from the gates.
cfg.plan.target_steps = None
cfg.plan.competitor_steps = None  # derived: relax + phonons (harmonic free energies of the hull); add "md" for MD folders in every competing phase too
cfg.plan.md_temperatures  = None  # None -> the 'temperature' of each run-list entry; or e.g. [300, 600]
cfg.plan.prune_window_eV  = None  # optional cost saver, e.g. 0.15: competitors that far above the 0 K hull get no phonons/MD

# -- Relaxation (used when "relax" is in cfg.plan.gates) ------------------------------------------------
cfg.relax.fmax = 0.005              # eV/A
cfg.relax.max_steps = 1000
cfg.relax.relax_cell = True         # relax cell shape and volume too (False: ions only, the given cell is kept)
cfg.relax.constant_volume = False   # keep the volume, relax shape + ions
cfg.relax.fix_symmetry = True       # keep the space group during the relaxation
cfg.relax.symprec = 1e-3
cfg.relax.pressure_GPa = 0.0

# -- Harmonic phonons (phonopy, finite displacements) ---------------------------------------------------
# Sizing: None = "as needed". An explicit supercell wins (then leave min_length_A None); otherwise the smallest
# supercell with perpendicular widths >= min_length_A is built (None -> built-in 12 A); max_atoms None = no limit.
cfg.phonon.supercell = None         # e.g. (3, 3, 3)
cfg.phonon.min_length_A = None      # e.g. 12.0
cfg.phonon.max_atoms = None         # e.g. 800 as a guard against runaway cells
cfg.phonon.delta_A = 0.01           # displacement amplitude
cfg.phonon.mesh_length_A = 80.0     # q-mesh density for the DOS
cfg.phonon.band_npoints = 61
cfg.phonon.symprec = 1e-4
cfg.phonon.imag_tol_meV = 0.5       # imaginary modes shallower than this count as numerical noise
cfg.phonon.fc_order = 2             # 3 = also third-order constants (phono3py, very slow); then add "fc3" to the steps

# -- Third-order force constants (only with "fc3" in the steps) -----------------------------------------
cfg.fc3.supercell = None            # e.g. (2, 2, 2); same rule as above (None -> built-in 8 A)
cfg.fc3.min_length_A = None
cfg.fc3.max_atoms = None            # e.g. 400
cfg.fc3.delta_A = 0.03
cfg.fc3.cutoff_pair_A = None        # ignore atom pairs farther than this (e.g. 4.0): far fewer displaced cells
cfg.fc3.separate_fc2 = True         # fc2 from the harmonic supercell (cfg.phonon), not the small fc3 one
cfg.fc3.kappa_mesh_length_A = 30.0
cfg.fc3.temperatures = [100.0, 200.0, 300.0, 400.0, 600.0, 800.0, 1000.0]
cfg.fc3.min_freq_THz = 0.5
cfg.fc3.ratio_warn = 0.05

# -- QHA (only with "qha" in the steps) -----------------------------------------------------------------
cfg.qha.n_volumes = 7               # odd, >= 5 recommended
cfg.qha.volume_range = 0.06         # +/- fraction of V0
cfg.qha.poly_order = 4              # polynomial order of the F(V) fit at each T
cfg.qha.relax_internal = True       # relax ions at every volume (required)
cfg.qha.temperatures = list(range(0, 1501, 50))   # G(T) grid; the hull grid of notebook 03 must lie inside it

# -- Elastic tensor -------------------------------------------------------------------------------------
cfg.elastic.strain = 0.004
cfg.elastic.relax_ions = True       # relaxed-ion constants (clamped ions over-stiffen ferroelectrics)
cfg.elastic.fmax = 0.002

# -- MD -------------------------------------------------------------------------------------------------
cfg.md.ensemble = "NPT"  # 'NPT' (cell fluctuates) | 'NVT' (fixed cell)
cfg.md.timestep_fs = 1.0
cfg.md.n_steps = 300  # 30 ps at 1 fs; the analysis wants >= gates.min_production_ps after equilibration
cfg.md.nblock = 5  # a frame every 5 steps -> VDOS Nyquist 413 meV (20 would alias O-containing oxides)
cfg.md.ttime_fs = 50.0  # thermostat time constant
cfg.md.ptime_fs = 100.0  # barostat time constant (NPT)
cfg.md.pressure_GPa = 0.0
cfg.md.supercell = (2, 2, 2)  # e.g. (4, 4, 4); explicit wins, min_cell_A then stays None
cfg.md.min_cell_A = None  # minimum supercell edge for MD; None -> built-in 15 A
cfg.md.max_atoms = None  # e.g. 1500
cfg.md.seed = 12345

# -- Run list -------------------------------------------------------------------------------------------
USE_QUERY_FILE = False
QUERY_FILE = "next_candidates.json"   # [{"material": ..., "symmetry": ..., "temperature": ...}, ...]
MANUAL_LIST = [                       # "temperature": operating T and (unless cfg.plan.md_temperatures) MD T; may be a list
    {"material": "BaTiO3", "symmetry": "mp-1076932", "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5020",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5777",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-644497",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-995191",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5933",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5986",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-504715",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-558125",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-19990",   "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-2998",    "temperature": 300},
]

## Prepare
A POSCAR in `database/<material>/<label>/` is used as is; one found only under `input/candidates/` is copied into the
database. Competing phases of every chemical subsystem are looked up in the database and in the candidates; the ones from
the candidates are copied into the database, and a `plan.json` plus the folders of all planned calculations are written for
everything. Competing phases that are already
in the database with other settings are **refused** (see `prepared.refused`). Safe to repeat: when an option changed,
notebook 02 sees the affected steps as *stale*.

In [ ]:
run_list = json.load(open(QUERY_FILE)) if USE_QUERY_FILE else MANUAL_LIST
prepared = prepare_project(run_list)            # competitors=False: targets only
if prepared.missing:
    print("missing structures:", prepared.missing)
for tag, conflicts in prepared.refused.items():       # already in the database, calculated with other settings: not used
    print("REFUSED", tag, {k: f"{a} -> {b}" for k, (a, b) in conflicts.items()})

11 target(s), 40 competing phase(s) prepared


## What will be computed
Supercells are taken from the *unrelaxed* structure (they can change by a repetition after relaxation) - check them before spending GPU time.

In [ ]:
display(pd.DataFrame(preview_plan(prepared)).set_index("phase"))
show_status()           # everything is pending until notebook 02 (or your own code) has run

,role,source,steps,operating_T_K,md_T_K,phonons cell,md cell
phase,,,,,,,
BaTiO3/mp-1076932,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x3x2 (180 atoms),2x2x2 (80 atoms)
BaTiO3/mp-5020,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x3x2 (270 atoms),2x2x2 (120 atoms)
BaTiO3/mp-5777,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x3x4 (360 atoms),2x2x2 (80 atoms)
BaTiO3/mp-644497,target,database,"relax, elastic, phonons, md",300.0,[300.0],2x2x4 (160 atoms),2x2x2 (80 atoms)
BaTiO3/mp-995191,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x3x3 (270 atoms),2x2x2 (80 atoms)
BaTiO3/mp-5933,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x3x1 (270 atoms),2x2x2 (240 atoms)
BaTiO3/mp-5986,target,database,"relax, elastic, phonons, md",300.0,[300.0],4x4x3 (240 atoms),2x2x2 (40 atoms)
BaTiO3/mp-504715,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x3x3 (135 atoms),2x2x2 (40 atoms)
BaTiO3/mp-558125,target,database,"relax, elastic, phonons, md",300.0,[300.0],3x2x1 (360 atoms),2x2x2 (480 atoms)


states: pending 124
  [PENDING   ] Ba/mp-1008283: md@300K
  [PENDING   ] Ba/mp-1008283: relax
  [PENDING   ] Ba/mp-1058581: md@300K
  [PENDING   ] Ba/mp-1058581: relax
  [PENDING   ] Ba/mp-10679: md@300K
  [PENDING   ] Ba/mp-10679: relax
  [PENDING   ] Ba/mp-1096835: md@300K
  [PENDING   ] Ba/mp-1096835: relax
  [PENDING   ] Ba/mp-1096840: md@300K
  [PENDING   ] Ba/mp-1096840: relax
  [PENDING   ] Ba/mp-1977763: md@300K
  [PENDING   ] Ba/mp-1977763: relax
  [PENDING   ] Ba/mp-570384: md@300K
  [PENDING   ] Ba/mp-570384: relax
  [PENDING   ] Ba/mp-605790: md@300K
  [PENDING   ] Ba/mp-605790: relax
  [PENDING   ] BaTiO3/mp-1076932: elastic
  [PENDING   ] BaTiO3/mp-1076932: md@300K
  [PENDING   ] BaTiO3/mp-1076932: phonons
  [PENDING   ] BaTiO3/mp-1076932: relax
  [PENDING   ] BaTiO3/mp-19990: elastic
  [PENDING   ] BaTiO3/mp-19990: md@300K
  [PENDING   ] BaTiO3/mp-19990: phonons
  [PENDING   ] BaTiO3/mp-19990: relax
  [PENDING   ] BaTiO3/mp-2998: elastic
  [PENDING   ] BaTiO3/mp-2998: md

In [ ]:
save_project()          # heatup_project.json: the folder roots, so notebooks 02 and 03 find this project

'heatup_project.json'

## The folders
One folder per planned calculation, each with a `task_<step>.json`. To compute a step with **another code** (e.g. DFT), do it in that
folder and write the files listed under `outputs` (the layout and the fields notebook 03 reads are in its last cell); notebook 02 is
then not needed for that step, and notebook 03 treats the results as `external`.

In [ ]:
print_layout(cfg.phases.database_root, max_depth=3)
print(open(os.path.join(cfg.phases.database_root, prepared.targets[0].tag, "relaxation", "task_relax.json")).read())

database/
├── Ba/
│   ├── mp-1008283/
│   │   ├── md/
│   │   ├── relaxation/
│   │   ├── POSCAR  (360 B)
│   │   ├── metadata.json  (98 B)
│   │   └── plan.json  (1.0 kB)
│   ├── mp-1058581/
│   │   ├── md/
│   │   ├── relaxation/
│   │   ├── POSCAR  (502 B)
│   │   ├── metadata.json  (98 B)
│   │   └── plan.json  (1.0 kB)
│   ├── mp-10679/
│   │   ├── md/
│   │   ├── relaxation/
│   │   ├── POSCAR  (502 B)
│   │   ├── metadata.json  (96 B)
│   │   └── plan.json  (1.0 kB)
│   ├── mp-1096835/
│   │   ├── md/
│   │   ├── relaxation/
│   │   ├── POSCAR  (786 B)
│   │   ├── metadata.json  (98 B)
│   │   └── plan.json  (1.0 kB)
│   ├── mp-1096840/
│   │   ├── md/
│   │   ├── relaxation/
│   │   ├── POSCAR  (502 B)
│   │   ├── metadata.json  (98 B)
│   │   └── plan.json  (1.0 kB)
│   ├── mp-1977763/
│   │   ├── md/
│   │   ├── relaxation/
│   │   ├── POSCAR  (289 B)
│   │   ├── metadata.json  (98 B)
│   │   └── plan.json  (1.0 kB)
│   ├── mp-570384/
│   │   ├── md/
│   │   ├── relaxation/
│